# 02 · Factor research

The B5 scoreboard. Every factor has to earn its place with a number
before it is allowed into the composite.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
pd.set_option('display.width', 140)

# Synthetic market today. When Matt's A3 cache exists, this becomes:
#     from quant.factors.base import load_panel
#     panel = load_panel()
from tests.conftest import make_synthetic_panel
panel, truth = make_synthetic_panel()
panel


In [ ]:
from quant.factors.market import default_market_factors
from quant.signals.cross_sectional import build_factor_panels, rebalance_dates
from quant.alpha.statistical_tests import factor_scoreboard, signal_decay

factors = default_market_factors()
dates = rebalance_dates(panel, warmup=truth['first_signal_row'])
panels = build_factor_panels(factors, panel, dates)
print(f'{len(factors)} factors over {len(dates)} rebalance dates')


## Scoreboard

Read `mean_ic` in context: 0.02-0.05 is a real equity factor, 0.10+ deserves
a leakage check, 0.30+ is a bug. `t_stat` above ~2 is the usual bar.


In [ ]:
board = factor_scoreboard(factors, panel, horizon=21, method='spearman', dates=dates)
board[['factor','category','n','mean_ic','ir','t_stat','p_value','hit_rate']]


## Are these four factors saying the same thing?

This is the question that decides IC weighting vs ridge. Highly correlated
factors double-count the same bet, and IC weighting cannot see that.


In [ ]:
stacked = pd.DataFrame({n: df.stack(future_stack=True) for n, df in panels.items()}).dropna()
stacked.corr().round(3)


## Decay

Shape decides how it is traded. Fast decay means costs eat it; a flat or
rising curve is a slow signal you can hold at a monthly rebalance.


In [ ]:
decay = signal_decay(panels['momentum_12_1'], panel, horizons=(1,5,10,21,63), name='momentum_12_1')
decay[['mean_ic','t_stat','hit_rate','overlapping']]


## Quantile spread

The tradeable version of IC: what the top decile returned against the bottom.


In [ ]:
from quant.signals.cross_sectional import forward_returns
scores, fwd = panels['momentum_12_1'], forward_returns(panel, 21, dates)
rows = []
for d in scores.index:
    s, r = scores.loc[d].dropna(), fwd.loc[d]
    if len(s) < 20: continue
    q = pd.qcut(s, 5, labels=False, duplicates='drop')
    rows.append(r.groupby(q).mean())
spread = pd.DataFrame(rows).mean()
print(spread.apply(lambda x: f'{x:+.2%}'))
print(f'\ntop minus bottom quintile: {spread.iloc[-1] - spread.iloc[0]:+.2%} per 21 days')
